# Phase 4 Colab execution launcher

This notebook only verifies the runtime and invokes the existing Phase 4 CLI; it contains no training implementation. The approved config, G3 manifests, and preprocessed arrays are copied unchanged to `/content/blockchain-phase4`. Outputs are written to the mounted Google Drive directory.

Inputs may be supplied either as `/content/drive/MyDrive/phase4-colab-inputs.zip` or as an extracted folder `/content/drive/MyDrive/phase4-colab-inputs/` whose contents start at the project root (`configs/`, `data/`, `src/`, ...). Every file is checked against the recorded SHA-256 manifest.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
import torch

gpu_check = shutil.which("nvidia-smi")
if gpu_check is None:
    raise RuntimeError(
        "nvidia-smi is not on PATH. This kernel is not a Colab GPU runtime "
        f"(python={sys.version.split()[0]}, executable={sys.executable}). "
        "Select Kernel -> Colab -> Auto Connect, then pick the Colab runtime."
    )
gpu_check = subprocess.run([gpu_check], capture_output=True, text=True)
if gpu_check.returncode != 0:
    raise RuntimeError(f"nvidia-smi failed: {gpu_check.stderr}")
if not torch.cuda.is_available():
    raise RuntimeError("The connected Colab runtime does not expose CUDA")
print(gpu_check.stdout)
print({"python": sys.version, "torch": torch.__version__, "cuda": torch.version.cuda, "device": torch.cuda.get_device_name(0)})

In [ ]:
from google.colab import drive
import hashlib
import zipfile
from pathlib import Path

drive.mount("/content/drive")
BUNDLE_ZIP = Path("/content/drive/MyDrive/phase4-colab-inputs.zip")
BUNDLE_DIR = Path("/content/drive/MyDrive/phase4-colab-inputs")
EXPECTED_ZIP_SHA256 = "8b7e8548ff12a3882ef5cde3543e1998fc76c027c14537fb89e74e744123da7f"
EXPECTED_FILES = {
    ".gitignore": "c67e2b23745877b7ae632b04a3fcb80d8be2b8156872200ae61b992278f5a9e7",
    "Literature%20Review.md": "0319335ee2556e5f5748bfd9ba78f421a644175e317dbd35eb4d30a5ba4fe5ad",
    "MASTER_RESEARCH_EXECUTION_SPECIFICATION.md": "b2f4d1a128e0245616f0d1e38b4e69895bb53bfaaa8f98a47943bd93e17be381",
    "README.md": "9771af93413f94ca9212adadb38736b59e1fd26d092aeae26d3b41ef0146ca87",
    "configs/README.md": "d8c1875236da31ba63f47d912d97ddafedb37af39c46291f8a7bb045195a631c",
    "configs/data/aami5_mapping_v1.yaml": "ab41e2982dd83f73906cf98ae6b89cea7addf9dd34ed8ef8d7a0605cb957e0c4",
    "configs/federated/phase5_f1_fedavg_iid_like.yaml": "f3932c2295fafd423fdeb4283ad1a72c28ef0a5c2bf80cef1b3c882172fe4bd6",
    "configs/federated/phase5_f2_fedavg_moderate.yaml": "1df848c434ba456e72f2fac2361c4e6baf28024c5b13cadde28100c535881e75",
    "configs/federated/phase5_f3_fedavg_severe.yaml": "481850fb6fe0515ae81bf302dcc7a4683a4371dbffb997f0db537484b134afd8",
    "configs/federated/phase5_f4_fedprox_severe.yaml": "458fb29d7c10a861b9a071246a631641cf6d26350f41df05214621f0c36c553e",
    "configs/federated/phase5_f5_fedprox_moderate.yaml": "260b60db1d3d7190e9fada384a14b6ec830bd706642121384e1ab2a92f6fb687",
    "configs/models/phase4_cnn_v1.yaml": "3d03ff3993b7c9c3fb8bab57f90a65b94fb2690c7e63ce0962ac1706f648593b",
    "configs/models/phase4_diag_d1_class_weighted.yaml": "02ddbcdc583fe7a364041f6ab5dbdb3e6b09be99b8bcb6a64be0fb808f864a1f",
    "configs/models/phase4_diag_d2_selection_balanced_accuracy.yaml": "0a74724a2f7d05bd21b1586c058f3339bc75efc7d4c4a6bc8b2d1c06559fc497",
    "data/processed/phase3/test_record_ids.npy": "597114a395f46fe49b98c1b6285a95b13aea9f0afd7412360ab91ed08ceac2ee",
    "data/processed/phase3/test_sample_positions.npy": "949012b57297eec106ca56f80ffec00caace93df70fac425d131044df9d824cc",
    "data/processed/phase3/test_symbols.npy": "1e5e108780aa4556586e6f874f35c7961a7931bd31cf6c987b814950b186f5fd",
    "data/processed/phase3/test_x.npy": "4cdbe57cc38fe68fa73b05d9289fce1746c8a045244ff5f74ca350dc6bdb2ea1",
    "data/processed/phase3/test_y.npy": "917214af98a018f44d9a7da19351c20c3fc6a9407694a53f103902e9a6deb351",
    "data/processed/phase3/train_record_ids.npy": "b91dbf3b4297087818436c9d947078b77916edcdd18dfec3dd2b67d1e84fb12f",
    "data/processed/phase3/train_sample_positions.npy": "3a29c173e2edf641b71eec6422971f0010c8626f27d3e0842f6ec68603f19233",
    "data/processed/phase3/train_symbols.npy": "e7abea61fce78ddd2b96c70a8bcc3499e83e63f44a76a20c202ad4632170643e",
    "data/processed/phase3/train_x.npy": "b56c9673e342e854a439f5363f23d6ab95877bdfacb7eac8ce2c4c6894ea94bd",
    "data/processed/phase3/train_y.npy": "acec6c8b19458e0db0067fadd94feaf4b9b983356b01084692df874caa67e200",
    "data/processed/phase3/validation_record_ids.npy": "20866cb00e15b68fd7c6d09b5dd20186c037ca16ac280b7b18a4651b335a9617",
    "data/processed/phase3/validation_sample_positions.npy": "4dd6c71325fa66182615dcad30cdcc0348de1f4dbf5d21e9211e0aff4634cf2b",
    "data/processed/phase3/validation_symbols.npy": "56536153f7480ebd0cd1a0ede294a01fa8dc59a9b5fa6e5e6ac20620a02efcf1",
    "data/processed/phase3/validation_x.npy": "00386c0a4ca95057ae2f6cf02ab17a191cef18afd280df32e191afb293a9129d",
    "data/processed/phase3/validation_y.npy": "3405555802bea20d59d6282289739e8397f7cdf217abf4e46d4b7fe2e49c1957",
    "data/splits/client_manifests.json": "2f09bfaa54b0868f234fcd215008312ca254e3ddf3ab679832b06cd15338a95b",
    "data/splits/phase3_split_manifest.json": "64046b52e7825bab0155169b474cbcc22a0f78308a3ebff61e71d41bca73bf61",
    "data/splits/phase5/index.json": "ae53d5d0ec1cdf5b2409c73fa1cfc5716a7311bb066789345b16c66a7557d364",
    "data/splits/phase5/phase5_f1_iid_like.json": "9b7349592605719edb472cc1cf3aae87d50143d3960491a1d6cba3bc40fbc45c",
    "data/splits/phase5/phase5_f2_moderate.json": "a80b53207292731149c06d904feeef5499232accca0cd8b14d4893ca9e875924",
    "data/splits/phase5/phase5_f3_severe.json": "5a6eec6973dd405c250e747f4bca8e65cf1dbb081c8d34d6af9c989cbbfb1ea6",
    "docs/PHASE4_BASELINE_PROTOCOL.md": "fe0d4ea6982d77763a05b7534daca21932267216371a9ef957311c1dd18edf8e",
    "docs/PHASE4_DIAGNOSTIC_ABLATION_RESULTS.md": "0313763253d11184f21926fe13b67ba6e81103c916085540bd684ba76fd62963",
    "docs/PHASE4_EXECUTION_STATUS.md": "a839167f2cdb32fd2d6a11e39f86c58bfcea6c927ef167572d33ba76a3c3185c",
    "docs/PHASE_3_FINAL_EVALUATION.md": "4b09ddd73bd503459b824f56275afbc8cfb92d4043872aeb5bc5fc6ef10a76d3",
    "docs/PHASE_4_DIAGNOSTIC_ABLATION.md": "11051422c2962f81287d87bd1f9aa41aa11a0f5ef5935fa3ccd9e1809aec96b5",
    "docs/PHASE_4_S_CLASS_FAILURE_INVESTIGATION.md": "912ca018a77ed664df2b618f6a0be519c2c56f1f4056dd1103a460d076ebabb6",
    "docs/PHASE_5_EXECUTION_STATUS.md": "93d939de01594828c83bd8d3f04b8099a705a199f5a45d2f24a8452cc3c6b085",
    "docs/PHASE_5_IMPLEMENTATION_PLAN.md": "a302262e25b8cf31fa7916560bd75cbeedcd3ce3beea3dd022f19bb0be4fd80f",
    "docs/PHASE_5_MANIFEST_AUDIT.md": "3a0893034b110288ad4d178743846231b949a98beec801cf41692ad4736ba262",
    "docs/PROJECT_STARTUP_AUDIT.md": "41240e4533695588f8cc3f7eee7de21a33cdee4e89df3069b663ba5ed38b0c2f",
    "docs/data_and_labels/AAMI_MAPPING_AND_EXCLUSIONS.md": "a8541fab18d940bac635d96a9389cd80f48b40a62802ed1a64d87a5476fc3cc2",
    "docs/data_and_labels/DATA_PROVENANCE.md": "30e9018b59d09facf9769f69c8796af183774f8eefc54262fbd5283fafeeed10",
    "docs/data_and_labels/MITBIH_AUDIT_SUMMARY.json": "717105db8789fdb1baa275095551aca738b86d906cd7b6a73d940e94e500b688",
    "docs/data_and_labels/MITBIH_PER_RECORD_AAMI_COUNTS.csv": "6e9167f7f96d509070e34cfb376ae652f254205c0312ddd7898970fec2f1d230",
    "docs/data_and_labels/MITBIH_RAW_ANNOTATION_INVENTORY.csv": "8c12c89031aa6454421b3db2b37de238cd53875e25905d8d2e68001a544f6624",
    "docs/data_and_labels/MITBIH_v1.0.0_SHA256SUMS.csv": "01475af50c75514541495cde522b4c7ef2374d001e04f96c031e560924e5545d",
    "docs/data_and_labels/PHASE3_PREPROCESSING_AUDIT.json": "3f4d8d8c2c337d4b5fffa794a795e0331e96701616fa088ebf12e5a5e7146ef8",
    "docs/data_and_labels/SPLIT_POLICY.md": "72650fe92dd68493c7a1c3c97d1eb70b4b79dacab59d6f4ac4750ac678122e01",
    "docs/limitations/LIMITATIONS.md": "9c182c441e6e8054b76ccbcf465f477834dc4fb69881d58f77f8ea03e36c34e4",
    "docs/literature/LITERATURE_REVIEW.md": "fb6d9d2f077646a872b4b1bc423bfc95ffd0d4af8685c94731163839b4195222",
    "docs/literature/REFERENCES.md": "db6aebb7c60263f45d04ddcd6f155a6189542461b0861afda9d0cc1f33046117",
    "docs/methodology/EXPERIMENT_REGISTRY.md": "f3db7ef1bda215a3212f5214d8a7ab877e933c59c88b8b1ee56326582078b6c9",
    "docs/methodology/RESULTS_AND_ANALYSIS.md": "6d937744c3268d8d6bdceb649796a34ebb37e5dae49fc549e826387061be44e6",
    "docs/protocol/RESEARCH_PROTOCOL.md": "fbfcf786d66b77199475ee9c1f97fea48e89814d654f1821051113bfea478b25",
    "docs/reproducibility/REPRODUCIBILITY.md": "a3781d5c19a9650cfec4a89c4305d36791a9777bb05dac68d679945b88752176",
    "docs/threat_model/PRIVACY_THREAT_MODEL.md": "c7d09bccf027a82889738acda4a23dbb0651069a8d0a1e3e8fc46effdccfba34",
    "docs/threat_model/SECURITY_THREAT_MODEL.md": "2a75dc6c5ac6dc2954230341e1b00bcce9fc8aa1d9bcf8413ac9a3e47df1320c",
    "experiments/README.md": "8599c3923b673903461a630c9fdf15a9162f74fc1eb744a16eb5f74a12ea2021",
    "experiments/s_class_investigation.py": "ee6028670277fcb3c3d8a6ec8738d0bc04c021b0581265cbb6ec8b5fa17de897",
    "experiments/s_class_record_contrast.py": "ffa2ab9a6238b37258d2599561a3825e44df5a25a94b7cc140a685690378924f",
    "experiments/s_class_separability.py": "821706d4a788fbd6964da297d703c40d5c815e4cbe7d3a4e543ab0c2f2e7f93a",
    "pyproject.toml": "9d9ca06964a7471c69245aac9569489cc38ca2648b4ca290e04e42399b5c42e7",
    "requirements-lock.txt": "ee803e22bf721099b3bb08f112167c3b52183796fa50b38fc421560efa7cdc99",
    "scripts/README.md": "993985f63eae23c0c32be45de91aeef945723ced30b5617f6c7e34df60c6912b",
    "scripts/audit_mitdb_annotations.py": "8b6f304a4ae076e191c409ece4ef65a675c51e824e30abf09ffcb35948640af2",
    "scripts/build_phase5_partitions.py": "ecc152a79927c31212918f54a8fc2556ccaa240bfa8015dbb2b0becf89d72588",
    "scripts/create_phase3_splits.py": "7a88be875d8cb6286039c963dc696cd4bb33e2e2645c224720882b6904c08188",
    "scripts/preprocess_mitdb_phase3.py": "a52b6688c24b616a770453428dcd1caa79c3f3a59dbfba6ad848dd7d373f69d2",
    "scripts/run_phase4_baselines.py": "35866bc86c667718252735ee6df7886e2e980b8ca181c581c470e138d44d38d0",
    "scripts/run_phase4_diagnostics.py": "20d67a578ce32b8bcc2fa92b79e1c85de199ac85a0b32f8916518c372f8da62c",
    "scripts/run_phase5_federated.py": "2c86997134fa43221cb78af0e5a9a013ffc831d905c7b983fed0ee5665e4bd30",
    "src/__init__.py": "7021c8f3dbd87d853980888d3b6cd422b03a1803495881067bfda657fb5b8fc6",
    "src/aggregation/__init__.py": "34f80c72fad6ef1b8bd3ca4f3f79e79e94ed8210ea7f78459b8de171565e514d",
    "src/blockchain/__init__.py": "a3aae8b5b189ba543f79cf4e7ff8bd666dd005b89257bfaf6967d6e1f7bd52c6",
    "src/data/__init__.py": "026d1fb09a94c64a07972589b6518ec4915424d8a5dc709cdba1d3ef6a17f084",
    "src/data/aami_mapping.py": "5531d83742dfef45491a562c285d8528768081ada2e0581ff399948b9d3665a9",
    "src/data/phase3_pipeline.py": "b87a57ea5ef15bc10cfa7b2db39d29a0f0dbefd81fe9d90cadde86e027b9f3ed",
    "src/evaluation/__init__.py": "29b80fa560273f6f179f1afa77a6987c0123a77bed813c2c0bb1a1ec58fb16ba",
    "src/evaluation/phase4_metrics.py": "a33e7d6ac07cb97b5a71fec31043baccbda3e58e0317f342f09b018a3244ab0b",
    "src/federated/__init__.py": "4d20c316ef99c6c5d2df6c8da228ee21d912e05c22af8bd7a1f3eaa61060fc52",
    "src/federated/fedavg.py": "00438221b93d8a875880baee7f0d550c42efd1c36b8bcf083e46c7ecb7f3b1c7",
    "src/federated/partitions.py": "95e798826e9800a048e26e7e2c97fb1e495a2c6a294db3870647d57755597896",
    "src/models/__init__.py": "f47c864811aa89e8b1118de5972c6e9199db50419ce6fa2806e92e1570f943f1",
    "src/models/cnn1d.py": "7cff7ad131dd5081c9865992f61d36e9a816ff3a1d0d5dcd1239dfe212d730f6",
    "src/preprocessing/__init__.py": "51b600f34e702f132fc2bf6fc75fc9f9fcdb1fbd1837dfca8ec0bb06d7d55763",
    "src/privacy/__init__.py": "45f2aae76161bf02cef14ed20fcd49da68887bbd6985c4d51687fbf364169beb",
    "src/training/__init__.py": "7c8f1cf0b59b1ba5c3b9700338b98a1e40e4104eb2371715a92d41ea50aedf28",
    "src/training/phase4.py": "3c27d6b322e0ced4db359a122f83419c7bf38e33e3cddff23de7271144d54ffc",
    "src/training/phase4_data.py": "8b60609186b037702ad183efcfb4043a472d1560580492183184f8f0a9ae76c8",
    "src/training/phase4_diagnostics.py": "70704996f7cb47c08f27de9bba5401a6e02660b538447a0a13a002c983ed9099",
    "src/utils/__init__.py": "0daed2a07ef5213e03f4719ace4c83f1fbe5be2537484eaed6286f9c35dfea65",
    "tests/__init__.py": "843ca0ba3ff7e034ddf12a01dae5e1196bf9bf01a45ded481eed66e857d5d371",
    "tests/test_aami_mapping.py": "f894ac0e8b51bd4d1e073c1006f0ad715c8b57272d0937c47c83e4dc4fe625ea",
    "tests/test_phase3_pipeline.py": "811969568f60bc1a571cf994abf10dfd1387cea68303b5f6ff1065984ec2235b",
    "tests/test_phase4_data.py": "dbf66d9d5b24dc6080a1ff7fe9bc712da5a63c37babadb0c7505ab9bdbd72785",
    "tests/test_phase4_diagnostics.py": "d31d3a9d9a63ffc3138788f8c6555423ffd22375b8ac33a77a203935fb54b11b",
    "tests/test_phase4_metrics.py": "d478a835b6d65c7e7626593dfb2a8364a91f75a8643b5a69c2d09c2b485fb1c3",
    "tests/test_phase4_model.py": "e967671649b6ceaa086de910bca42c13433662e1cde12292b063474a4d6fb208",
    "tests/test_phase4_training.py": "739f135dbafc85f22b07f5905937917a49f0cba8566e6e3ff088c7d563804363",
    "tests/test_phase5_federated.py": "9b4cef6ece92f54345fae17d6e61753d1fce196fa69a3f546e8125ebb9de9417",
    "tests/test_placeholder.py": "c7922541d6b9b457f9752a32fef00d0c5dd552d9eb37eb9f4d2ba68959d1c891"
}

if BUNDLE_ZIP.is_file():
    bundle_kind = "zip"
    archive_sha256 = hashlib.sha256(BUNDLE_ZIP.read_bytes()).hexdigest()
    if archive_sha256 != EXPECTED_ZIP_SHA256:
        raise ValueError(f"Input bundle hash mismatch: {archive_sha256}")
    with zipfile.ZipFile(BUNDLE_ZIP) as bundle:
        present = {m.filename.replace(chr(92), "/") for m in bundle.infolist()}
    missing = sorted(set(EXPECTED_FILES) - present)
    if missing:
        raise FileNotFoundError("Bundle entries missing: " + ", ".join(missing[:10]))
    print({"bundle": "zip", "sha256": archive_sha256, "entries": len(present)})
elif BUNDLE_DIR.is_dir():
    bundle_kind = "folder"
    missing = [name for name in EXPECTED_FILES if not (BUNDLE_DIR / name).is_file()]
    if missing:
        raise FileNotFoundError(
            "Drive folder is missing files (upload the project root contents, not a wrapped subfolder): "
            + ", ".join(missing[:10])
        )
    mismatched = [
        name for name, digest in EXPECTED_FILES.items()
        if hashlib.sha256((BUNDLE_DIR / name).read_bytes()).hexdigest() != digest
    ]
    if mismatched:
        raise ValueError("Drive folder files do not match the recorded hashes: " + ", ".join(mismatched[:10]))
    print({"bundle": "folder", "files": len(EXPECTED_FILES), "all_hashes_match": True})
else:
    raise FileNotFoundError(f"Upload {BUNDLE_ZIP} or the extracted folder {BUNDLE_DIR}")

In [ ]:
import hashlib
import shutil
import zipfile
from pathlib import Path, PureWindowsPath

target_root = Path("/content/blockchain-phase4")
existing_files = [str(p) for p in target_root.rglob("*") if p.is_file()] if target_root.exists() else []
if existing_files:
    raise FileExistsError("Refusing to overwrite existing files in /content/blockchain-phase4: " + ", ".join(existing_files[:10]))

if bundle_kind == "zip":
    with zipfile.ZipFile(BUNDLE_ZIP) as bundle:
        bad_member = bundle.testzip()
        if bad_member is not None:
            raise RuntimeError(f"Corrupt archive member: {bad_member}")
        for member in bundle.infolist():
            member_name = member.filename.replace(chr(92), "/")
            parts = tuple(part for part in member_name.split("/") if part not in ("", "."))
            windows_path = PureWindowsPath(member.filename)
            if "\x00" in member.filename or member_name.startswith("/") or not parts or ".." in parts or windows_path.is_absolute() or windows_path.drive:
                raise RuntimeError(f"Archive contains an unsafe path: {member.filename!r}")
            destination = target_root.joinpath(*parts)
            if member.is_dir() or member_name.endswith("/"):
                destination.mkdir(parents=True, exist_ok=True)
                continue
            destination.parent.mkdir(parents=True, exist_ok=True)
            with bundle.open(member) as source, destination.open("xb") as output:
                shutil.copyfileobj(source, output)
else:
    for name in EXPECTED_FILES:
        destination = target_root / name
        destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(BUNDLE_DIR / name, destination)

PROJECT_ROOT = target_root
required_paths = [PROJECT_ROOT / name for name in EXPECTED_FILES if name.startswith(("configs/models/", "data/splits/", "src/training/phase4", "docs/data_and_labels/PHASE3_PREPROCESSING_AUDIT.json"))]
missing_paths = [str(p) for p in required_paths if not p.is_file()]
if missing_paths:
    raise FileNotFoundError("Required project inputs missing after copy: " + ", ".join(missing_paths))
print({"bundle_kind": bundle_kind, "project_root": str(PROJECT_ROOT), "copied_files": len(EXPECTED_FILES)})

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/blockchain-phase4")
OUTPUT_ROOT = Path("/content/drive/MyDrive/blockchain/experiments/logs/phase4/colab_full_20261113")
smoke_dir = OUTPUT_ROOT / "gpu_smoke"
full_run_dir = OUTPUT_ROOT / "full_run"
print({"project_root": str(PROJECT_ROOT), "drive_output_root": str(OUTPUT_ROOT)})

BASELINE_CONFIG = PROJECT_ROOT / "configs/models/phase4_cnn_v1.yaml"


def run_phase4_cli(extra_args, module="scripts.run_phase4_baselines"):
    argv = [sys.executable, "-m", module, "--project-root", str(PROJECT_ROOT)]
    if module == "scripts.run_phase4_baselines":
        argv += ["--config", str(BASELINE_CONFIG)]
    argv += list(extra_args)
    print({"argv": argv, "python": sys.executable})
    result = subprocess.run(argv, cwd=PROJECT_ROOT, text=True, capture_output=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        raise SystemExit(f"Phase 4 CLI exited {result.returncode}")


print({"interpreter": sys.executable, "torch": torch.__version__})


In [ ]:
import shutil

if smoke_dir.exists():
    leftover = sorted(str(path.relative_to(smoke_dir)) for path in smoke_dir.rglob("*") if path.is_file())
    print({"leftover_files": leftover})
    if leftover:
        raise SystemExit("gpu_smoke contains files from a previous run; move or delete it before continuing")
    shutil.rmtree(smoke_dir)
run_phase4_cli(["--output-dir", str(smoke_dir), "--smoke-test", "--device", "cuda"])
print({"smoke_test": (smoke_dir / "smoke_test.json").read_text()})


Inspect the GPU smoke output above before starting the full sweep. The full run executes five centralized and 25 local-only fits and refuses to reuse an existing output directory. If it is interrupted, run the resume cell against the same `full_run` directory.

In [ ]:
import shutil

if full_run_dir.exists():
    if any(full_run_dir.iterdir()):
        raise SystemExit("full_run already has content; use the resume cell instead of starting a new run")
    shutil.rmtree(full_run_dir)
run_phase4_cli(["--output-dir", str(full_run_dir), "--device", "cuda"])


In [ ]:
run_phase4_cli(["--output-dir", str(full_run_dir), "--device", "cuda", "--resume"])


## Phase 4 diagnostic ablation (approved Option 1)

These cells run the two pre-registered diagnostics described in
`docs/PHASE_4_DIAGNOSTIC_ABLATION.md`. They do not modify the frozen baseline config,
its 30 runs, checkpoints, or the G3 split. Each writes to its own Drive directory and
refuses to overwrite an existing one. Test data is never used for selection or tuning.


In [ ]:
import shutil
from pathlib import Path

DIAGNOSTIC_FILES = [
    "configs/models/phase4_diag_d1_class_weighted.yaml",
    "configs/models/phase4_diag_d2_selection_balanced_accuracy.yaml",
    "src/training/phase4_diagnostics.py",
    "scripts/run_phase4_diagnostics.py",
]

synced = []
for name in DIAGNOSTIC_FILES:
    source = BUNDLE_DIR / name
    if not source.is_file():
        raise FileNotFoundError(
            "Diagnostic file missing from the Drive bundle: " + name
            + ". Re-upload the phase4-colab-inputs folder from the project root."
        )
    destination = PROJECT_ROOT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, destination)
    synced.append(name)
print({"synced": synced})


In [ ]:
import shutil
from datetime import datetime, timezone

DIAG_ROOT = Path("/content/drive/MyDrive/blockchain/experiments/logs/phase4/diagnostics")
EXPERIMENTS = ["phase4_diag_d1_class_weighted", "phase4_diag_d2_selection_balanced_accuracy"]

for experiment in EXPERIMENTS:
    out = DIAG_ROOT / experiment / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    if out.exists():
        raise SystemExit("Diagnostic output directory already exists; rerun to get a new timestamp")
    run_phase4_cli(
        ["--experiment", experiment, "--output-dir", str(out)],
        module="scripts.run_phase4_diagnostics",
    )
    print({"experiment": experiment, "output": str(out)})


In [ ]:
import json
from pathlib import Path

baseline = json.loads((full_run_dir / "summary.json").read_text())
base_test = baseline["aggregate_test_metrics"]["centralized"]["test"]
base_val = baseline["aggregate_test_metrics"]["centralized"]["validation"]

DIAG_ROOT = Path("/content/drive/MyDrive/blockchain/experiments/logs/phase4/diagnostics")


def fmt(value, digits=3):
    return "NA" if value is None else f"{value:.{digits}f}"


def supported(block):
    order = block["confusion_matrix"]["class_order"]
    support = block["confusion_matrix"]["true_support"]
    return [name for name, n in zip(order, support) if n > 0]


def per_class_line(block, tag):
    order = block["confusion_matrix"]["class_order"]
    support = block["confusion_matrix"]["true_support"]
    pieces = []
    for name, n in zip(order, support):
        cell = block["per_class"][name]
        pieces.append(
            f"{name}[rec {fmt(cell['recall']['mean'])} "
            f"pre {fmt(cell['precision']['mean'])} "
            f"f1 {fmt(cell['f1']['mean'])} n={n}]"
        )
    print(f"  {tag}: " + " ".join(pieces))


def show(name, val_block, test_block):
    v, t = val_block["overall"], test_block["overall"]
    print(name)
    print(f"  val  acc {fmt(v['accuracy']['mean'], 4)}  macroF1 {fmt(v['macro_f1']['mean'], 4)}  included {supported(val_block)}")
    print(f"  test acc {fmt(t['accuracy']['mean'], 4)} sd {fmt(t['accuracy']['standard_deviation'], 4)}"
          f"  macroF1 {fmt(t['macro_f1']['mean'], 4)} sd {fmt(t['macro_f1']['standard_deviation'], 4)}"
          f"  wf1 {fmt(t['weighted_f1']['mean'], 4)}  included {supported(test_block)}")
    per_class_line(test_block, "test")


show("BASELINE phase4_cnn_v1 (frozen reference)", base_val, base_test)
for experiment in ["phase4_diag_d1_class_weighted", "phase4_diag_d2_selection_balanced_accuracy"]:
    candidates = sorted((DIAG_ROOT / experiment).glob("*/summary.json"))
    if not candidates:
        print(f"{experiment}: NO RESULTS")
        continue
    summary = json.loads(candidates[-1].read_text())
    print()
    show(experiment, summary["aggregate"]["validation"], summary["aggregate"]["test"])
    print("  weights:", {k: round(v, 4) for k, v in summary["derived_class_weights"].items()})
    print("  per-seed S recall:",
          [fmt(r["test_metrics"]["per_class"]["S"]["recall"]) for r in summary["runs"]])
    print("  per-seed test macroF1:",
          [fmt(r["test_metrics"]["macro_f1"]) for r in summary["runs"]])
    print("  per-seed best_epoch:",
          [r["best_epoch"] for r in summary["runs"]])
print()
print("Test F support is 0, so test F metrics are not estimable in every row above.")


## Phase 5 — Federated learning (approved Stage B)

Approved decisions: 50 communication rounds, 2 local epochs, all five clients every round,
Dirichlet alpha 0.5 (moderate) and 0.1 (severe) over record groups, FedProx proximal_mu 0.01,
BatchNorm buffers weighted by sample count, optimizer state reset each round.

Conditions: F1 IID-like, F2 moderate, F3 severe (FedAvg), F4 FedProx severe, F5 FedProx
moderate. Five seeds each. Validation selects the round; test is evaluated once per run.

This runs as a subprocess against the verified package. Expect this to take hours: run one
condition at a time and keep the Colab tab open.


In [ ]:
import shutil
from pathlib import Path

PHASE5_FILES = [
    "configs/federated/phase5_f1_fedavg_iid_like.yaml",
    "configs/federated/phase5_f2_fedavg_moderate.yaml",
    "configs/federated/phase5_f3_fedavg_severe.yaml",
    "configs/federated/phase5_f4_fedprox_severe.yaml",
    "configs/federated/phase5_f5_fedprox_moderate.yaml",
    "data/splits/phase5/index.json",
    "data/splits/phase5/phase5_f1_iid_like.json",
    "data/splits/phase5/phase5_f2_moderate.json",
    "data/splits/phase5/phase5_f3_severe.json",
    "src/federated/partitions.py",
    "src/federated/fedavg.py",
    "scripts/run_phase5_federated.py",
]

synced = []
for name in PHASE5_FILES:
    source = BUNDLE_DIR / name
    if not source.is_file():
        raise FileNotFoundError("Missing from Drive bundle: " + name)
    destination = PROJECT_ROOT / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copyfile(source, destination)
    synced.append(name)
print({"synced_phase5_files": len(synced)})


In [ ]:
import hashlib
import json

EXPECTED_PARTITION_HASHES = {
    "phase5_f1_iid_like": "7d635f51c966fd47dbfa8db960aa27f23d66dc514dc28113842bf2271e54c0a2",
    "phase5_f2_moderate": "c447d732cb9e886b87db10b50bec8ec3df9944458198162f920b8966413a02d3",
    "phase5_f3_severe": "da8b96fa118e53bf53bee50b4f56bb6c281c65093b3c690058664f7b0cf8bbd4",
}
for name, expected in EXPECTED_PARTITION_HASHES.items():
    path = PROJECT_ROOT / "data/splits/phase5" / f"{name}.json"
    document = json.loads(path.read_text())
    if document["manifest_sha256"] != expected:
        raise ValueError(f"Partition manifest hash mismatch for {name}")
    counts = {c["client_id"]: c["class_counts"]["S"] for c in document["clients"]}
    print(name, document["manifest_sha256"][:12], "S per client", counts)
print({"partition_integrity": "verified"})


In [ ]:
import shutil
from datetime import datetime, timezone

PHASE5_ROOT = Path("/content/drive/MyDrive/blockchain/experiments/logs/phase5")
CONDITION = "phase5_f1_fedavg_iid_like"  # change per run; one condition at a time
SEED = None  # None runs all five seeds

out = PHASE5_ROOT / CONDITION / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if out.exists():
    raise SystemExit("Phase 5 output directory already exists")
args = ["--experiment", CONDITION, "--output-dir", str(out), "--device", "cuda"]
if SEED is not None:
    args += ["--seed", str(SEED)]
run_phase4_cli(args, module="scripts.run_phase5_federated")


In [ ]:
CONDITION = "phase5_f1_fedavg_iid_like"
SEED = 42
OUT = "/content/drive/MyDrive/blockchain/experiments/logs/phase5/<run_timestamp>"
run_phase4_cli(
    ["--experiment", CONDITION, "--output-dir", OUT, "--device", "cuda",
     "--seed", str(SEED), "--resume"],
    module="scripts.run_phase5_federated",
)


In [ ]:
import json
from pathlib import Path

PHASE5_ROOT = Path("/content/drive/MyDrive/blockchain/experiments/logs/phase5")
for condition_dir in sorted(PHASE5_ROOT.glob("phase5_*")):
    runs = sorted(condition_dir.glob("*/seed_*/run_summary.json"))
    if not runs:
        continue
    print(condition_dir.name)
    for run_path in runs:
        summary = json.loads(run_path.read_text())
        test = summary["test_metrics"]
        print(
            f"  seed {summary['seed']:>2} round {summary['selected_round']:>3} "
            f"acc {test['accuracy']:.4f} macroF1 {test['macro_f1']:.4f} "
            f"S_recall {test['per_class']['S']['recall']} wf1 {test['weighted_f1']:.4f}"
        )
